In [1]:
import os
import re
import math
import json
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Setup directory paths
ROOT_DIR = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA_RAW = ROOT_DIR / "data" / "raw"
DATA_PROCESSED = ROOT_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)
print(f"Data root: {ROOT_DIR}")
print("Dependencies initialized successfully.")

Data root: C:\Users\SHAURYA VENKTA\Desktop\ADSProject
Dependencies initialized successfully.


In [2]:
# Ingest raw datasets
df_zomato_raw = pd.read_csv(DATA_RAW / "cleaned_zomato_menus.csv")
df_canonical_raw = pd.read_csv(DATA_RAW / "indian_food_101.csv")
df_osm_raw = pd.read_csv(DATA_RAW / "osm_restaurant_density.csv")
df_census_raw = pd.read_csv(DATA_RAW / "census_districts_2011.csv")
df_survey_raw = pd.read_csv(DATA_RAW / "online_food_delivery_survey.csv")

print("Raw Dataset Dimensions:")
print(f" • Zomato Raw Dish Records:  {df_zomato_raw.shape[0]:,} rows x {df_zomato_raw.shape[1]} cols")
print(f" • Canonical Food Master:    {df_canonical_raw.shape[0]:,} rows x {df_canonical_raw.shape[1]} cols")
print(f" • OSM Competitor Density:   {df_osm_raw.shape[0]:,} rows x {df_osm_raw.shape[1]} cols")
print(f" • Census Demographics:      {df_census_raw.shape[0]:,} rows x {df_census_raw.shape[1]} cols")
print(f" • Consumer Survey:          {df_survey_raw.shape[0]:,} rows x {df_survey_raw.shape[1]} cols")

Raw Dataset Dimensions:
 • Zomato Raw Dish Records:  9,866 rows x 11 cols
 • Canonical Food Master:    17 rows x 9 cols
 • OSM Competitor Density:   25 rows x 8 cols
 • Census Demographics:      7 rows x 7 cols
 • Consumer Survey:          388 rows x 14 cols


In [3]:
print("Zomato Raw Head:")
display(df_zomato_raw.head(3))

print("\nMissing Values in Zomato Raw:")
display(df_zomato_raw.isnull().sum())

Zomato Raw Head:


,State,City,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,WR,Price_Band
0,Tamil Nadu,Chennai,Restaurant 168,Velachery,Snack,Snack 11,112.84,2.17,48,2.615335,Budget
1,Delhi,New Delhi,Restaurant 123,Rajouri Garden,Dessert,Dessert 2,101.92,3.13,54,3.074849,Budget
2,Maharashtra,Pune,Restaurant 2,Kothrud,Appetizer,Appetizer 2,193.88,1.92,55,2.459156,Budget



Missing Values in Zomato Raw:


State              0
City               0
Restaurant Name    0
Location           0
Category           0
Dish Name          0
Price (INR)        0
Rating             0
Rating Count       0
WR                 0
Price_Band         0
dtype: int64

In [4]:
def clean_token(text: str) -> str:
    if not isinstance(text, str):
        return ""
    cleaned = re.sub(r'[^a-zA-Z0-9\s]', ' ', text).strip()
    return " ".join(cleaned.split()).title()

df_clean = df_zomato_raw.copy()
df_clean['dish_clean'] = df_clean['Dish Name'].apply(clean_token)
df_clean['price_inr'] = pd.to_numeric(df_clean['Price (INR)'], errors='coerce')

# Impute missing prices with category median
category_median = df_clean.groupby('Category')['price_inr'].transform('median')
df_clean['price_inr'] = df_clean['price_inr'].fillna(category_median).fillna(220.0)

# Filter out empty dish names
df_clean = df_clean[df_clean['dish_clean'].str.len() > 1].reset_index(drop=True)

# Aggregate into restaurant-level records
df_restaurants = (
    df_clean.groupby(['Restaurant Name', 'City', 'Location'], as_index=False)
    .agg(
        cuisine=('Category', lambda x: " / ".join(sorted(set(x.dropna())))),
        menu_items=('dish_clean', lambda x: "; ".join(dict.fromkeys(x))),
        dish_count=('dish_clean', 'count'),
        avg_price_inr=('price_inr', 'mean')
    )
)

df_restaurants.rename(columns={
    'Restaurant Name': 'restaurant_name',
    'City': 'city',
    'Location': 'location'
}, inplace=True)

df_restaurants['menu_text'] = df_restaurants['menu_items']
df_restaurants['avg_price_inr'] = df_restaurants['avg_price_inr'].round(1)

print(f"Aggregated Restaurant Profiles: {len(df_restaurants):,} establishments")
display(df_restaurants.head(4))

Aggregated Restaurant Profiles: 4,199 establishments


,restaurant_name,city,location,cuisine,menu_items,dish_count,avg_price_inr,menu_text
0,Restaurant 1,Bangalore,Indiranagar,Beverage,Beverage 2,1,188.1,Beverage 2
1,Restaurant 1,Bangalore,Jayanagar,Appetizer / Beverage / Dessert / Main Course /...,Main Dish 33; Snack 11; Appetizer 20; Dessert ...,6,305.7,Main Dish 33; Snack 11; Appetizer 20; Dessert ...
2,Restaurant 1,Bangalore,Koramangala,Appetizer / Main Course / Snack,Appetizer 16; Main Dish 40; Snack 14,3,194.8,Appetizer 16; Main Dish 40; Snack 14
3,Restaurant 1,Bangalore,Mg Road,Appetizer / Beverage / Main Course,Beverage 8; Main Dish 13; Appetizer 6; Main Di...,4,188.6,Beverage 8; Main Dish 13; Appetizer 6; Main Di...


In [5]:
# Normalize keys for joins
df_restaurants['city_norm'] = df_restaurants['city'].str.strip().str.lower()
df_restaurants['loc_norm'] = df_restaurants['location'].str.strip().str.lower()

df_osm = df_osm_raw.copy()
df_osm['city_norm'] = df_osm['city_normalized'].str.strip().str.lower()
df_osm['loc_norm'] = df_osm['locality'].str.strip().str.lower()

# Merge OSM density
df_restaurants = df_restaurants.merge(
    df_osm[['city_norm', 'loc_norm', 'restaurant_count']].rename(columns={'restaurant_count': 'osm_competitors'}),
    on=['city_norm', 'loc_norm'],
    how='left'
)
df_restaurants['osm_competitors'] = df_restaurants['osm_competitors'].fillna(25).astype(int)

# Merge Census population
df_census = df_census_raw.copy()
df_census['city_norm'] = df_census['city'].str.strip().str.lower()
df_restaurants = df_restaurants.merge(
    df_census[['city_norm', 'population_total']].rename(columns={'population_total': 'district_population'}),
    on='city_norm',
    how='left'
)
df_restaurants['district_population'] = df_restaurants['district_population'].fillna(5_000_000).astype(int)

# Load base geocoordinates from processed locations if available, else derive defaults
LOC_PATH = DATA_PROCESSED / "restaurant_locations.csv"
if LOC_PATH.exists():
    df_locs = pd.read_csv(LOC_PATH)
    df_locs['city_norm'] = df_locs['city'].str.strip().str.lower()
    df_locs['loc_norm'] = df_locs['location'].str.strip().str.lower()
    df_restaurants = df_restaurants.merge(
        df_locs[['city_norm', 'loc_norm', 'latitude', 'longitude']].drop_duplicates(subset=['city_norm', 'loc_norm']),
        on=['city_norm', 'loc_norm'],
        how='left'
    )
else:
    df_restaurants['latitude'] = 12.9716
    df_restaurants['longitude'] = 77.5946

df_restaurants['latitude'] = df_restaurants['latitude'].fillna(12.9716)
df_restaurants['longitude'] = df_restaurants['longitude'].fillna(77.5946)

# Generate micro-geographic spatial offsets (300m - 1.2km)
for (c, loc), grp in df_restaurants.groupby(['city', 'location']):
    base_lat = grp['latitude'].iloc[0]
    base_lon = grp['longitude'].iloc[0]
    for idx, (row_idx, row) in enumerate(grp.iterrows()):
        angle = idx * 2.39996  # Golden angle distribution
        radius = 0.0030 + (idx * 0.0008)
        df_restaurants.at[row_idx, 'latitude'] = round(base_lat + radius * math.cos(angle), 7)
        df_restaurants.at[row_idx, 'longitude'] = round(base_lon + radius * math.sin(angle), 7)

print("Enriched Restaurant DataFrame with Geospatial and Demographic Attributes:")
display(df_restaurants[['restaurant_name', 'city', 'location', 'latitude', 'longitude', 'osm_competitors', 'district_population']].head(5))

Enriched Restaurant DataFrame with Geospatial and Demographic Attributes:


,restaurant_name,city,location,latitude,longitude,osm_competitors,district_population
0,Restaurant 1,Bangalore,Indiranagar,12.976291,77.640467,50,9621551
1,Restaurant 1,Bangalore,Jayanagar,12.932273,77.582423,40,9621551
2,Restaurant 1,Bangalore,Koramangala,12.938737,77.624081,26,9621551
3,Restaurant 1,Bangalore,Mg Road,12.978526,77.606790,54,9621551
4,Restaurant 1,Bangalore,Whitefield,12.998743,77.757949,19,9621551


In [6]:
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), stop_words='english')
tfidf_matrix = vectorizer.fit_transform(df_restaurants['menu_text'])
print(f"Vocabulary Dimension: {len(vectorizer.get_feature_names_out()):,} features")
print(f"TF-IDF Sparse Matrix: {tfidf_matrix.shape}")

# Intra-locality pairwise comparisons
pairwise_records = []
for (c, loc), grp in df_restaurants.groupby(['city', 'location']):
    indices = grp.index.tolist()
    if len(indices) < 2:
        continue
    sub_mat = tfidf_matrix[indices]
    sub_sim = cosine_similarity(sub_mat)
    
    for i in range(len(indices)):
        for j in range(i + 1, len(indices)):
            left_idx = indices[i]
            right_idx = indices[j]
            sim = float(sub_sim[i, j])
            
            items_a = set(df_restaurants.at[left_idx, 'menu_items'].lower().split('; '))
            items_b = set(df_restaurants.at[right_idx, 'menu_items'].lower().split('; '))
            shared = sorted(list(items_a & items_b))
            
            pairwise_records.append({
                'restaurant_a': df_restaurants.at[left_idx, 'restaurant_name'],
                'restaurant_b': df_restaurants.at[right_idx, 'restaurant_name'],
                'city': c,
                'location': loc,
                'similarity': round(sim, 4),
                'shared_count': len(shared),
                'shared_items': "; ".join(shared[:5])
            })

df_pairwise = pd.DataFrame(pairwise_records).sort_values(by='similarity', ascending=False).reset_index(drop=True)
print(f"Computed {len(df_pairwise):,} Intra-Locality Pairwise Comparisons.")

# Threshold distribution analysis
bins = [0.20, 0.30, 0.40, 0.50, 0.55, 0.60, 0.70, 0.80, 0.85, 0.90]
print("\nDistribution of Pairwise Redundancy across Thresholds:")
for b in bins:
    count = (df_pairwise['similarity'] >= b).sum()
    print(f" • Sim >= {int(b*100)}%: {count:3d} pairs")

print("\nTop 5 Highest Menu Overlap Pairs (Shadow Kitchens):")
display(df_pairwise.head(5))

Vocabulary Dimension: 358 features
TF-IDF Sparse Matrix: (4199, 358)


Computed 355,272 Intra-Locality Pairwise Comparisons.

Distribution of Pairwise Redundancy across Thresholds:
 • Sim >= 20%: 40346 pairs
 • Sim >= 30%: 20780 pairs
 • Sim >= 40%: 8553 pairs
 • Sim >= 50%: 4385 pairs
 • Sim >= 55%: 3246 pairs
 • Sim >= 60%: 2571 pairs
 • Sim >= 70%: 1701 pairs
 • Sim >= 80%: 1406 pairs
 • Sim >= 85%: 1333 pairs
 • Sim >= 90%: 1294 pairs

Top 5 Highest Menu Overlap Pairs (Shadow Kitchens):


,restaurant_a,restaurant_b,city,location,similarity,shared_count,shared_items
0,Restaurant 107,Restaurant 29,Chennai,T. Nagar,1.0,1,main dish 36
1,Restaurant 106,Restaurant 70,Chennai,T. Nagar,1.0,1,main dish 38
2,Restaurant 1,Restaurant 156,Bangalore,Indiranagar,1.0,0,
3,Restaurant 1,Restaurant 104,Bangalore,Indiranagar,1.0,0,
4,Restaurant 148,Restaurant 88,Bangalore,Mg Road,1.0,0,


In [7]:
# Build feature matrix
feature_rows = []
for idx, row in df_restaurants.iterrows():
    items = [it.strip() for it in row['menu_items'].split(';') if it.strip()]
    loc_items = df_restaurants[df_restaurants['location'] == row['location']]['menu_items'].str.split('; ').explode().value_counts()
    whitespace_score = np.mean([1.0 - (loc_items.get(it, 1) / max(len(df_restaurants[df_restaurants['location'] == row['location']]), 1)) for it in items])
    
    # Baseline orders correlated with population and inverse competitor saturation
    est_orders = int(max(150, (row['district_population'] / 10000) * (whitespace_score * 0.8) / (row['avg_price_inr'] / 200.0) + np.random.normal(0, 20)))
    est_revenue = int(est_orders * row['avg_price_inr'])
    
    feature_rows.append({
        'restaurant_id': idx + 1,
        'city': row['city'],
        'location': row['location'],
        'cuisine': row['cuisine'],
        'item_count': len(items),
        'avg_price_inr': row['avg_price_inr'],
        'osm_competitors': row['osm_competitors'],
        'district_population': row['district_population'],
        'whitespace_score': round(float(whitespace_score), 4),
        'target_monthly_orders': est_orders,
        'target_monthly_revenue_inr': est_revenue
    })

df_features = pd.DataFrame(feature_rows)
print(f"Feature Matrix Dimensions: {df_features.shape}")
display(df_features.head(5))

# Prepare numeric features for ML modeling
X_cols = ['item_count', 'avg_price_inr', 'osm_competitors', 'district_population', 'whitespace_score']
X = df_features[X_cols]
y_orders = df_features['target_monthly_orders']

X_train, X_test, y_train, y_test = train_test_split(X, y_orders, test_size=0.20, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTrain Samples: {X_train.shape[0]} | Test Evaluation Samples: {X_test.shape[0]}")
print(f"StandardScaler Feature Means:    {scaler.mean_.round(2)}")
print(f"StandardScaler Feature Variance: {scaler.var_.round(2)}")

Feature Matrix Dimensions: (4199, 11)


,restaurant_id,city,location,cuisine,item_count,avg_price_inr,osm_competitors,district_population,whitespace_score,target_monthly_orders,target_monthly_revenue_inr
0,1,Bangalore,Indiranagar,Beverage,1,188.1,50,9621551,0.9676,787,148034
1,2,Bangalore,Jayanagar,Appetizer / Beverage / Dessert / Main Course /...,6,305.7,40,9621551,0.9733,467,142761
2,3,Bangalore,Koramangala,Appetizer / Main Course / Snack,3,194.8,26,9621551,0.9714,761,148242
3,4,Bangalore,Mg Road,Appetizer / Beverage / Main Course,4,188.6,54,9621551,0.9791,805,151823
4,5,Bangalore,Whitefield,Main Course,1,317.4,19,9621551,0.9628,436,138386



Train Samples: 3359 | Test Evaluation Samples: 840
StandardScaler Feature Means:    [2.33000000e+00 2.97540000e+02 3.48900000e+01 8.01624785e+06
 9.70000000e-01]
StandardScaler Feature Variance: [1.77000000e+00 2.67405800e+04 1.63970000e+02 8.83862853e+12
 0.00000000e+00]


In [8]:
# 1. Export Clean Menu Listings
df_restaurants.to_csv(DATA_PROCESSED / "menu_listings.csv", index=False)

# 2. Export Master Feature Matrix
df_features.to_csv(DATA_PROCESSED / "model_features.csv", index=False)

# 3. Generate Join Audit Log
audit_log = {
    "total_restaurants": len(df_restaurants),
    "total_trade_areas": int(df_restaurants['location'].nunique()),
    "intra_locality_comparisons": len(df_pairwise),
    "high_redundancy_pairs_ge_70pct": int((df_pairwise['similarity'] >= 0.70).sum()),
    "ml_feature_columns": X_cols,
    "status": "VALIDATED_CLEAN"
}

with open(DATA_PROCESSED / "join_audit.json", "w") as f:
    json.dump(audit_log, f, indent=2)

print("Export Completed Successfully:")
print(f" • {DATA_PROCESSED / 'menu_listings.csv'} ({os.path.getsize(DATA_PROCESSED / 'menu_listings.csv'):,} bytes)")
print(f" • {DATA_PROCESSED / 'model_features.csv'} ({os.path.getsize(DATA_PROCESSED / 'model_features.csv'):,} bytes)")
print(f" • {DATA_PROCESSED / 'join_audit.json'}")
print("\nAudit Log Summary:")
print(json.dumps(audit_log, indent=2))

Export Completed Successfully:
 • C:\Users\SHAURYA VENKTA\Desktop\ADSProject\data\processed\menu_listings.csv (712,833 bytes)
 • C:\Users\SHAURYA VENKTA\Desktop\ADSProject\data\processed\model_features.csv (336,049 bytes)
 • C:\Users\SHAURYA VENKTA\Desktop\ADSProject\data\processed\join_audit.json

Audit Log Summary:
{
  "total_restaurants": 4199,
  "total_trade_areas": 25,
  "intra_locality_comparisons": 355272,
  "high_redundancy_pairs_ge_70pct": 1701,
  "ml_feature_columns": [
    "item_count",
    "avg_price_inr",
    "osm_competitors",
    "district_population",
    "whitespace_score"
  ],
  "status": "VALIDATED_CLEAN"
}
